In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os
import time
import json
import pickle
import warnings
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    matthews_corrcoef
)

warnings.filterwarnings('ignore')

DRIVE_PATH = '/content/drive/MyDrive/CICIoT2023_Research'
os.makedirs(f'{DRIVE_PATH}/models_cat8_ml', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/results_cat8_ml', exist_ok=True)

CATEGORY_ENCODING = {
    'DDoS': 0,
    'DoS': 1,
    'Mirai': 2,
    'Benign': 3,
    'Recon': 4,
    'Spoofing': 5,
    'Web': 6,
    'BruteForce': 7
}
ID_TO_CATEGORY = {v: k for k, v in CATEGORY_ENCODING.items()}
target_names = [ID_TO_CATEGORY[i] for i in range(8)]

All imports OK
XGBoost version: 3.2.0


In [ ]:
print("Loading multiclass data...")

X_train = pd.read_csv(f'{DRIVE_PATH}/X_train_balanced_8cat_smoteenn.csv')
y_train = pd.read_csv(f'{DRIVE_PATH}/y_train_balanced_8cat_smoteenn.csv').squeeze()

X_val = pd.read_csv(f'{DRIVE_PATH}/X_val_cat8.csv')
y_val = pd.read_csv(f'{DRIVE_PATH}/y_val_cat8.csv').squeeze()

X_test = pd.read_csv(f'{DRIVE_PATH}/X_test_cat8.csv')
y_test = pd.read_csv(f'{DRIVE_PATH}/y_test_cat8.csv').squeeze()

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("\nTrain distribution:")
print(y_train.value_counts().sort_index())

Category encoding:
  0: DDoS
  1: DoS
  2: Mirai
  3: Recon
  4: Spoofing
  5: Web
  6: BruteForce
  7: Benign


In [ ]:
def compute_multiclass_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average='weighted'),
        "f1_macro": f1_score(y_true, y_pred, average='macro'),
        "mcc": matthews_corrcoef(y_true, y_pred)
    }

Label mapping loaded — 34 fine-grained classes.


In [ ]:
print("Training Random Forest...")
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

rf.fit(X_train, y_train)
rf_time = time.time() - t0

rf_val_pred = rf.predict(X_val)
rf_test_pred = rf.predict(X_test)

rf_metrics = compute_multiclass_metrics(y_test, rf_test_pred)

print("RF test metrics:", rf_metrics)
print("\nRF classification report:")
print(classification_report(y_test, rf_test_pred, target_names=target_names, digits=4))

Loading X_val (to get feature names)...
X_val: (2100516, 39)  (9.7s)


In [ ]:
print("Training XGBoost...")
t0 = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_time = time.time() - t0

xgb_val_pred = xgb_model.predict(X_val)
xgb_test_pred = xgb_model.predict(X_test)

xgb_metrics = compute_multiclass_metrics(y_test, xgb_test_pred)

print("XGB test metrics:", xgb_metrics)
print("\nXGB classification report:")
print(classification_report(y_test, xgb_test_pred, target_names=target_names, digits=4))

Loading balanced training data...
X_train: (1629654, 39)  (7.7s)
y_train distribution (balanced):
  0 DDoS        :  310,101
  1 DoS         :  247,767
  2 Mirai       :  497,762
  3 Recon       :   72,947
  4 Spoofing    :  105,434
  5 Web         :   62,880
  6 BruteForce  :   32,083
  7 Benign      :  300,680


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, pred, title in zip(
    axes,
    [rf_test_pred, xgb_test_pred],
    ['Random Forest', 'XGBoost']
):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{title} — Test Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/results_cat8_ml/cat8_ml_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

y_val distribution (imbalanced):
  0 DDoS        : 1,228,858  (58.50%)
  1 DoS         :  417,978  (19.90%)
  2 Mirai       :  235,768  (11.22%)
  3 Recon       :   65,758  (3.13%)
  4 Spoofing    :   43,696  (2.08%)
  5 Web         :    2,490  (0.12%)
  6 BruteForce  :    1,237  (0.06%)
  7 Benign      :  104,731  (4.99%)


In [ ]:
results = [
    {
        'Model': 'Random Forest',
        'Accuracy': rf_metrics['accuracy'],
        'Balanced Accuracy': rf_metrics['balanced_accuracy'],
        'F1 Weighted': rf_metrics['f1_weighted'],
        'F1 Macro': rf_metrics['f1_macro'],
        'MCC': rf_metrics['mcc'],
        'Train Time(s)': rf_time
    },
    {
        'Model': 'XGBoost',
        'Accuracy': xgb_metrics['accuracy'],
        'Balanced Accuracy': xgb_metrics['balanced_accuracy'],
        'F1 Weighted': xgb_metrics['f1_weighted'],
        'F1 Macro': xgb_metrics['f1_macro'],
        'MCC': xgb_metrics['mcc'],
        'Train Time(s)': xgb_time
    }
]

results_df = pd.DataFrame(results)
results_df.to_csv(f'{DRIVE_PATH}/results_cat8_ml/cat8_ml_results.csv', index=False)
print(results_df)

Loading X_test and y_test (imbalanced — final evaluation only)...
X_test: (4201031, 39)  (21.5s)
y_test distribution (imbalanced — original):
  0 DDoS        : 2,458,206  (58.51%)
  1 DoS         :  836,061  (19.90%)
  2 Mirai       :  471,664  (11.23%)
  3 Recon       :  131,096  (3.12%)
  4 Spoofing    :   87,323  (2.08%)
  5 Web         :    4,695  (0.11%)
  6 BruteForce  :    2,524  (0.06%)
  7 Benign      :  209,462  (4.99%)


In [ ]:
pickle.dump(rf, open(f'{DRIVE_PATH}/models_cat8_ml/rf_cat8.pkl', 'wb'))
pickle.dump(xgb_model, open(f'{DRIVE_PATH}/models_cat8_ml/xgb_cat8.pkl', 'wb'))

print("Saved RF and XGB models.")

All sanity checks passed.
  Train: 1,629,654 rows (balanced)
  Val:   2,100,516 rows (imbalanced)
  Test:  4,201,031 rows (imbalanced)
